
# ver4からver4.2への変更点
- 最新版の av-19.0.1 がインストールされるようになったが、PyAVのバージョン19以降では av.open() 関数から metadata_errors という引数が削除された。
- しかし、現在の faster-whisper （バージョン1.2.1）は依然としてこの引数を指定して音声ファイルを読み込もうとするため、TypeError が発生して処理が止まってしまう。
### そこで、ライブラリのバージョンを無理やり落とす方法で解決を図った。

In [ ]:
!pip install -q faster-whisper "av<19.0.0" nvidia-cublas-cu12 nvidia-cudnn-cu12

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import ctypes
from datetime import datetime, timezone
import glob
import os
import shutil
import time
import zipfile

# Google Colab環境固有の問題（ライブラリパスの不一致）を解決するためのワークアラウンド
# Pythonパッケージ内のNVIDIA製GPUライブラリ(.so)を検索し、ctypesでメモリに直接ロードする
for so_file in glob.glob(
    '/usr/local/lib/python*/dist-packages/nvidia/*/lib/*.so*'
):
    try:
        ctypes.CDLL(so_file)
    except Exception:
        # 依存関係の順序などで読み込めないものはスキップ
        pass

from faster_whisper import WhisperModel

# ==========================================
# 設定情報の定義
# ==========================================
# 読み込み・保存先のGoogle Driveフォルダ（環境に合わせて変更してください）
DRIVE_FOLDER = '/content/drive/MyDrive/YourMeetingFolder/Data'

# ファイル名やチャットログ内のユーザー名を任意の表示名に変換するマッピング表
USER_NAME_MAPPING = {
    'AAA': 'Aさん',
    'BBB': 'Bさん',
    'CCC': 'Cさん',
}

# 文字起こしモデル特有のハルシネーション（AIの幻覚による定型文）を除外するためのブラックリスト
HALLUCINATION_BLACKLIST = [
    'ご視聴ありがとうございました',
    'チャンネル登録',
    '高評価',
    '概要欄',
    '字幕：',
    'お借りした',
    'BGM：',
    '視聴ありがとうございました',
]

# ==========================================
# ユーティリティ関数
# ==========================================
def get_mapped_name(raw_name):
    """登録されたユーザー名をマッピング表に基づいて変換する"""
    for key, new_name in USER_NAME_MAPPING.items():
        if key in raw_name:
            return new_name
    return raw_name

def format_relative_time(seconds):
    """秒数を HH:MM:SS または MM:SS の相対時間フォーマットに変換する"""
    is_negative = seconds < 0
    seconds = abs(seconds)
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    sign = '-' if is_negative else ''
    if h > 0:
        return f'{sign}{h:02d}:{m:02d}:{s:02d}'
    return f'{sign}{m:02d}:{s:02d}'

def is_hallucination(text):
    """テキストがハルシネーションのブラックリストに含まれているか判定する"""
    for black_word in HALLUCINATION_BLACKLIST:
        if black_word in text:
            return True
    return False

def get_craig_start_time(data_folder, chat_abs_times):
    """
    Craigボットが生成した info.txt から録音開始の絶対時間(UNIXタイムスタンプ)を取得する。
    見つからない場合はチャットログの最古の時間、または現在時刻を返す。
    """
    info_files = glob.glob(os.path.join(data_folder, '*info.txt'))
    if info_files:
        try:
            with open(info_files[0], 'r', encoding='utf-8') as f:
                for line in f:
                    if line.startswith('Start time:'):
                        time_str = line.split('Start time:')[1].strip()
                        time_str = time_str.replace('Z', '+00:00')
                        return datetime.fromisoformat(time_str).timestamp()
        except Exception as e:
            print(f'⚠️ info.txtの読み込みエラー: {e}')

    if chat_abs_times:
        return min(chat_abs_times)
    return time.time()

# ==========================================
# メイン処理
# ==========================================
def process_drive_data(drive_folder):
    """指定されたフォルダ内のZIP解凍、音声文字起こし、チャットログの統合を行う"""
    if not os.path.exists(drive_folder):
        print(f"❌ エラー: '{drive_folder}' が見つかりません。")
        return

    # 1. ZIPファイルの検索と一時フォルダへの展開
    zip_files = glob.glob(os.path.join(drive_folder, '*.zip'))
    target_folder = drive_folder

    if zip_files:
        # 最新のZIPファイルを対象に処理
        target_zip = max(zip_files, key=os.path.getctime)
        print(f'📦 ZIPファイルを検出しました: {os.path.basename(target_zip)}')
        print('🗜️ ZIPファイルを一時フォルダに展開中...')
        with zipfile.ZipFile(target_zip, 'r') as zip_ref:
            zip_ref.extractall(target_folder)
    else:
        print('⚠️ ZIPファイルが見つかりません。フォルダ内の音声ファイルを直接検索します。')

    # 2. Whisperモデルの初期化（GPU使用、高速化のためのfloat16指定）
    print('🤖 GPU(cuda)にAIモデルを読み込み中...')
    model = WhisperModel('large-v3-turbo', device='cuda', compute_type='float16')

    # 対象となる音声ファイルの検索
    extensions = ('*.m4a', '*.wav', '*.mp3', '*.aac', '*.flac')
    audio_files = []
    for ext in extensions:
        audio_files.extend(glob.glob(os.path.join(target_folder, ext)))

    if not audio_files:
        print('❌ エラー: 音声ファイルが見つかりません。')
        return

    print(f'📁 計 {len(audio_files)} 人分の音声を検出しました。精密タイムスタンプ解析を開始します。')

    all_timeline_items = []
    chat_abs_times = []
    start_time_proc = time.time()

    # 3. テキストチャットログの読み込み (音声との時系列統合用)
    chat_log_path = os.path.join(drive_folder, 'chat_log*.txt')
    if not os.path.exists(chat_log_path):
        chat_log_path = os.path.join(target_folder, 'chat_log.txt')

    if os.path.exists(chat_log_path):
        print('💬 チャットログを読み込み中...')
        with open(chat_log_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('|', 2)
                if len(parts) == 3:
                    abs_time = float(parts[0])
                    speaker = get_mapped_name(parts[1])
                    chat_abs_times.append(abs_time)
                    all_timeline_items.append({
                        'abs_time': abs_time,
                        'speaker': speaker,
                        'text': parts[2],
                        'type': 'chat',
                    })

    # 4. 音声とチャットを同期させるための基準時間(ゼロ地点)を取得
    craig_start_abs = get_craig_start_time(target_folder, chat_abs_times)

    # 5. 音声ファイルの文字起こし処理
    for file_path in audio_files:
        file_name = os.path.splitext(os.path.basename(file_path))[0]
        speaker_name = get_mapped_name(file_name)

        print(f'🎵 【{speaker_name}】 の音声を解析中...')
        # VAD(Voice Activity Detection)を有効にし、無音区間をスキップして効率化
        segments, _ = model.transcribe(
            file_path,
            beam_size=5,
            language='ja',
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=1000, speech_pad_ms=400),
            condition_on_previous_text=False,
        )

        for segment in segments:
            # 音声なし確率が高い（60%以上）セグメントは除外
            if hasattr(segment, 'no_speech_prob') and segment.no_speech_prob > 0.6:
                continue

            text = segment.text.strip()
            if text and not is_hallucination(text):
                all_timeline_items.append({
                    'abs_time': craig_start_abs + segment.start,
                    'speaker': speaker_name,
                    'text': text,
                    'type': 'voice',
                })

    # 6. 音声発言とテキストチャットを絶対時間でソートし、一つのタイムラインに統合
    print('🔄 発言とチャットを時系列順に統合しています...')
    all_timeline_items.sort(key=lambda x: x['abs_time'])

    # 7. 議事録テキストファイルとしての書き出し
    date_str = datetime.fromtimestamp(craig_start_abs).strftime('%Y%m%d')
    output_filename = f'merged_minutes_{date_str}.txt'
    output_dir = os.path.join(os.path.dirname(drive_folder), 'Meetingkaigiroku')
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, output_filename)

    with open(output_path, 'w', encoding='utf-8') as f:
        for item in all_timeline_items:
            rel_sec = item['abs_time'] - craig_start_abs
            rel_time_str = format_relative_time(rel_sec)
            local_time_str = datetime.fromtimestamp(item['abs_time']).strftime('%H:%M:%S')

            # [ローカル時刻 | 録音開始からの経過時間] のバッジを作成
            timestamp_badge = f'[{local_time_str} | {rel_time_str}]'

            if item['type'] == 'voice':
                f.write(f'{timestamp_badge} 【{item["speaker"]}】: {item["text"]}\n')
            else:
                f.write(f'{timestamp_badge}【{item["speaker"]} (Chat)】: {item["text"]}\n')

    elapsed = time.time() - start_time_proc
    print(f'✨ すべての処理が完了しました！ (処理時間: {elapsed:.1f}秒)')
    print(f'📄 出力完了: {output_path}')

# 処理の実行
if __name__ == "__main__":
    process_drive_data(DRIVE_FOLDER)